# Tables de régressions avec contrôles progressifs

Ce notebook reprend les régressions existantes du pipeline et ajoute les contrôles un par un pour voir comment évoluent les coefficients de traitement.

Couverture:
- DiD ADD sur Synchronicity et Idio_Vol
- DiD DELETE sur Synchronicity et Idio_Vol
- DML ADD sur Synchronicity
- DML DELETE sur Synchronicity, si l'échantillon est suffisant

In [7]:
from functools import lru_cache
from pathlib import Path
import re
import warnings

import doubleml as dml
import numpy as np
import pandas as pd
from IPython.display import display
from linearmodels.panel import PanelOLS
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 180)


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Impossible de localiser la racine du projet.')


def slugify(value: str) -> str:
    value = value.lower().strip()
    value = re.sub(r'[^a-z0-9]+', '_', value)
    return value.strip('_')


ROOT = find_project_root()
DATA_DIR = ROOT / 'data'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'
RESULTS_DIR = DATA_DIR / 'results'

PANEL_PATH = INTERMEDIATE_DIR / 'panel_monthly.parquet'
PRICES_PATH = INTERMEDIATE_DIR / 'prices_raw.parquet'
MATCHES_ADD_PATH = RESULTS_DIR / 'matched_pairs.csv'
MATCHES_DELETE_PATH = RESULTS_DIR / 'delete_matched_pairs.csv'

WINDOW_DID = 12
WINDOW_DML = 12
WINDOW_DAYS = 365
MIN_OBS = 200
N_FOLDS_DML = 5
N_REP_DML = 3
FEATURE_ORDER = ['Log_MarketCap', 'Momentum_12m', 'Volatility_pre']
DID_CONTROL_ORDER = ['Amihud', 'Avg_Volume']


print('Loading panel and price data ...')
panel = pd.read_parquet(PANEL_PATH)
panel['date'] = pd.to_datetime(panel['date'])

_p01 = panel['Amihud'].quantile(0.01)
_p99 = panel['Amihud'].quantile(0.99)
panel['Amihud'] = panel['Amihud'].clip(lower=_p01, upper=_p99)

_iv_p01 = panel['Idio_Vol'].quantile(0.01)
_iv_p99 = panel['Idio_Vol'].quantile(0.99)
panel['Idio_Vol'] = panel['Idio_Vol'].clip(lower=_iv_p01, upper=_iv_p99)

prices = pd.read_parquet(PRICES_PATH)
colmap = {c.lower(): c for c in prices.columns}
rename = {}
for target, variants in [
    ('close', ['close', 'adj close', 'adj_close']),
    ('volume', ['volume']),
    ('date', ['date']),
    ('ticker', ['ticker']),
]:
    for variant in variants:
        if variant in colmap:
            rename[colmap[variant]] = target
            break
prices = prices.rename(columns=rename)
prices['date'] = pd.to_datetime(prices['date'])
prices['close'] = pd.to_numeric(prices['close'], errors='coerce')
prices['volume'] = pd.to_numeric(prices['volume'], errors='coerce')
prices = prices[(prices['close'] > 0) & (prices['volume'] > 0)].copy()
prices.sort_values(['ticker', 'date'], inplace=True)
prices['log_ret'] = prices.groupby('ticker')['close'].transform(lambda s: np.log(s / s.shift(1)))
prices['log_ret'] = prices['log_ret'].replace([np.inf, -np.inf], np.nan)
prices = prices.dropna(subset=['log_ret']).copy()

matches_add = pd.read_csv(MATCHES_ADD_PATH)
matches_delete = pd.read_csv(MATCHES_DELETE_PATH)
for matches_df in [matches_add, matches_delete]:
    matches_df['event_date'] = pd.to_datetime(matches_df['event_date'])
    matches_df.drop(columns=[c for c in matches_df.columns if c.startswith('Unnamed:')], inplace=True, errors='ignore')
    matches_df = matches_df.loc[matches_df['match_valid'] == True].copy()

matches_add = matches_add[matches_add['match_valid'] == True].copy()
matches_delete = matches_delete[matches_delete['match_valid'] == True].copy()

print(f'Panel   : {len(panel):,} rows | {panel["ticker"].nunique()} tickers')
print(f'Prices  : {len(prices):,} rows | {prices["ticker"].nunique()} tickers')
print(f'ADD     : {len(matches_add):,} valid matched pairs')
print(f'DELETE  : {len(matches_delete):,} valid matched pairs')


def build_stacked_panel(panel_df: pd.DataFrame, matches_df: pd.DataFrame, window: int = WINDOW_DID) -> pd.DataFrame:
    stacked_rows = []
    pair_id = 0

    for _, match in matches_df.iterrows():
        ev_date = match['event_date']
        tk_treat = match['ticker_treated']
        tk_ctrl = match['ticker_control']

        date_start = ev_date - pd.DateOffset(months=window)
        date_end = ev_date + pd.DateOffset(months=window)

        for ticker, treat_val in [(tk_treat, 1), (tk_ctrl, 0)]:
            sub = panel_df[(panel_df['ticker'] == ticker) & (panel_df['date'] >= date_start) & (panel_df['date'] <= date_end)].copy()
            if len(sub) < 6:
                continue

            sub['pair_id'] = pair_id
            sub['Treat'] = treat_val
            sub['Post'] = (sub['date'] >= ev_date).astype(int)
            sub['Treat_Post'] = sub['Treat'] * sub['Post']
            sub['event_date'] = ev_date
            sub['rel_month'] = (sub['date'].dt.year - ev_date.year) * 12 + (sub['date'].dt.month - ev_date.month)
            sub['entity'] = f'{pair_id}_{ticker}'
            sub['time_id'] = sub['rel_month']
            stacked_rows.append(sub)

        pair_id += 1

    stacked = pd.concat(stacked_rows, ignore_index=True)
    stacked = stacked.dropna(subset=['Synchronicity', 'Idio_Vol', 'Amihud', 'Avg_Volume'])
    return stacked.set_index(['entity', 'time_id'])


@lru_cache(maxsize=None)
def compute_pre_event_features(event_date: pd.Timestamp, ticker: str, window_days: int = WINDOW_DAYS, min_obs: int = MIN_OBS):
    start = event_date - pd.Timedelta(days=window_days)
    sub = prices[(prices['ticker'] == ticker) & (prices['date'] >= start) & (prices['date'] < event_date)].copy()
    if len(sub) < min_obs:
        return None

    first_close = sub['close'].iloc[0]
    last_close = sub['close'].iloc[-1]
    mean_volume = sub['volume'].mean()
    vol_pre = sub['log_ret'].std()

    if first_close <= 0 or last_close <= 0 or mean_volume <= 0 or pd.isna(vol_pre):
        return None

    return {
        'Log_MarketCap': float(np.log(last_close * mean_volume)),
        'Momentum_12m': float(np.log(last_close / first_close)),
        'Volatility_pre': float(vol_pre),
    }


def build_dml_dataset(matches_df: pd.DataFrame, panel_df: pd.DataFrame, label: str, window_months: int = WINDOW_DML) -> pd.DataFrame:
    rows = []

    for _, match in matches_df.iterrows():
        ev_date = match['event_date']
        post_end = ev_date + pd.DateOffset(months=window_months)

        for ticker, treat_val in [(match['ticker_treated'], 1), (match['ticker_control'], 0)]:
            post_obs = panel_df[(panel_df['ticker'] == ticker) & (panel_df['date'] >= ev_date) & (panel_df['date'] <= post_end)]['Synchronicity'].dropna()
            if len(post_obs) < 3:
                continue

            feats = compute_pre_event_features(ev_date, ticker)
            if feats is None:
                continue

            rows.append(
                {
                    'event_type': label,
                    'ticker': ticker,
                    'event_date': ev_date,
                    'treated': treat_val,
                    'Synchronicity_post': float(post_obs.mean()),
                    **feats,
                }
            )

    dml_df = pd.DataFrame(rows).dropna()
    return dml_df


def star(p_value: float) -> str:
    if pd.isna(p_value):
        return ''
    if p_value < 0.01:
        return '***'
    if p_value < 0.05:
        return '**'
    if p_value < 0.1:
        return '*'
    return ''


def fit_panel_specs(stacked_df: pd.DataFrame, dep_var: str, controls_order: list[str], table_label: str):
    spec_defs = [controls_order[:i] for i in range(len(controls_order) + 1)]
    spec_labels = [f'M{i+1}' for i in range(len(spec_defs))]
    results = []

    for spec_label, controls in zip(spec_labels, spec_defs):
        x_cols = ['Treat_Post', *controls]
        y = stacked_df[dep_var]
        X = stacked_df[x_cols]
        mod = PanelOLS(y, X, entity_effects=True, time_effects=True, drop_absorbed=True)
        res = mod.fit(cov_type='clustered', cluster_entity=True)

        row = {
            'Model': spec_label,
            'Controls': ', '.join(controls) if controls else 'None',
            'N': int(res.nobs),
            'Within_R2': float(res.rsquared_within),
            'Treat_Post': float(res.params['Treat_Post']),
            'Treat_SE': float(res.std_errors['Treat_Post']),
            'Treat_p': float(res.pvalues['Treat_Post']),
            'Treat_Sig': star(res.pvalues['Treat_Post']),
        }

        for ctrl in controls_order:
            if ctrl in res.params.index:
                row[f'{ctrl}_coef'] = float(res.params[ctrl])
                row[f'{ctrl}_se'] = float(res.std_errors[ctrl])
                row[f'{ctrl}_p'] = float(res.pvalues[ctrl])
            else:
                row[f'{ctrl}_coef'] = np.nan
                row[f'{ctrl}_se'] = np.nan
                row[f'{ctrl}_p'] = np.nan

        results.append(row)

    out = pd.DataFrame(results)
    print('=' * 110)
    print(f'DiD table - {table_label} - {dep_var}')
    print('=' * 110)
    display(out[['Model', 'Controls', 'Treat_Post', 'Treat_SE', 'Treat_p', 'Treat_Sig', 'N', 'Within_R2']])

    out_path = RESULTS_DIR / f'regression_controls_did_{slugify(table_label)}_{slugify(dep_var)}.csv'
    out.to_csv(out_path, index=False)
    print(f'Exported: {out_path}')
    return out


def fit_dml_specs(dml_df: pd.DataFrame, table_label: str, feature_order: list[str] = FEATURE_ORDER):
    if len(dml_df) < 50:
        print(f'Skipping DML - {table_label}: not enough observations ({len(dml_df):,}).')
        return None

    spec_defs = [feature_order[:i] for i in range(1, len(feature_order) + 1)]
    spec_labels = [f'DML{i}' for i in range(1, len(spec_defs) + 1)]
    rows = []

    for spec_label, controls in zip(spec_labels, spec_defs):
        data_dml = dml.DoubleMLData(
            dml_df,
            y_col='Synchronicity_post',
            d_cols='treated',
            x_cols=controls,
        )
        plr = dml.DoubleMLPLR(
            data_dml,
            ml_l=RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            ml_m=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
            n_folds=N_FOLDS_DML,
            n_rep=N_REP_DML,
            score='partialling out',
        )
        plr.fit()

        theta = float(plr.coef[0])
        se = float(plr.se[0])
        pval = float(plr.pval[0])
        ci = plr.confint().iloc[0]
        rows.append(
            {
                'Model': spec_label,
                'Controls': ', '.join(controls),
                'N': int(len(dml_df)),
                'Treated': int(dml_df['treated'].sum()),
                'Theta': theta,
                'SE': se,
                'P_value': pval,
                'Sig': star(pval),
                'CI_low': float(ci.iloc[0]),
                'CI_high': float(ci.iloc[1]),
            }
        )

    out = pd.DataFrame(rows)
    print('=' * 110)
    print(f'DML table - {table_label}')
    print('=' * 110)
    display(out[['Model', 'Controls', 'Theta', 'SE', 'P_value', 'Sig', 'N', 'Treated']])

    out_path = RESULTS_DIR / f'regression_controls_dml_{slugify(table_label)}.csv'
    out.to_csv(out_path, index=False)
    print(f'Exported: {out_path}')
    return out


ROOT

Loading panel and price data ...
Panel   : 132,652 rows | 1033 tickers
Prices  : 2,891,560 rows | 1034 tickers
ADD     : 654 valid matched pairs
DELETE  : 599 valid matched pairs


PosixPath('/Users/ethan.bcht/Dev/master-thesis-code')

In [8]:
# Chargement des données et construction du panel empilé
WINDOW = 12

panel = pd.read_parquet(PANEL_PATH)
panel['date'] = pd.to_datetime(panel['date'])

_p01 = panel['Amihud'].quantile(0.01)
_p99 = panel['Amihud'].quantile(0.99)
panel['Amihud'] = panel['Amihud'].clip(lower=_p01, upper=_p99)

_iv_p01 = panel['Idio_Vol'].quantile(0.01)
_iv_p99 = panel['Idio_Vol'].quantile(0.99)
panel['Idio_Vol'] = panel['Idio_Vol'].clip(lower=_iv_p01, upper=_iv_p99)

matches = pd.read_csv(MATCHES_PATH)
matches['event_date'] = pd.to_datetime(matches['event_date'])
matches = matches[matches['match_valid'] == True].copy()

print(f'Panel: {len(panel):,} lignes | {panel["ticker"].nunique()} tickers')
print(f'Paires valides: {len(matches):,}')

def build_stacked_panel(panel_df: pd.DataFrame, matches_df: pd.DataFrame, window: int = 12) -> pd.DataFrame:
    stacked_rows = []
    pair_id = 0

    for _, match in matches_df.iterrows():
        ev_date = match['event_date']
        tk_treat = match['ticker_treated']
        tk_ctrl = match['ticker_control']

        date_start = ev_date - pd.DateOffset(months=window)
        date_end = ev_date + pd.DateOffset(months=window)

        for ticker, treat_val in [(tk_treat, 1), (tk_ctrl, 0)]:
            sub = panel_df[(panel_df['ticker'] == ticker) & (panel_df['date'] >= date_start) & (panel_df['date'] <= date_end)].copy()

            if len(sub) < 6:
                continue

            sub['pair_id'] = pair_id
            sub['Treat'] = treat_val
            sub['Post'] = (sub['date'] >= ev_date).astype(int)
            sub['Treat_Post'] = sub['Treat'] * sub['Post']
            sub['event_date'] = ev_date
            sub['rel_month'] = (sub['date'].dt.year - ev_date.year) * 12 + (sub['date'].dt.month - ev_date.month)
            sub['entity'] = f'{pair_id}_{ticker}'
            sub['time_id'] = sub['rel_month']

            stacked_rows.append(sub)

        pair_id += 1

    df = pd.concat(stacked_rows, ignore_index=True)
    df = df.dropna(subset=['Synchronicity', 'Idio_Vol', 'Amihud', 'Avg_Volume'])
    return df.set_index(['entity', 'time_id'])

df = build_stacked_panel(panel, matches, window=WINDOW)
print(f'Panel empilé: {len(df):,} lignes | {df.index.get_level_values(0).nunique()} entités')
df.head()

Panel: 132,652 lignes | 1033 tickers
Paires valides: 654
Panel empilé: 30,886 lignes | 1308 entités


date  ticker    R2_raw  Synchronicity  Idio_Vol    Amihud  Avg_Volume  N_obs  pair_id  Treat  Post  Treat_Post event_date  rel_month
entity   time_id                                                                                                                                           
0_AAK.ST -11     2014-12-01  AAK.ST  0.357217      -0.587464  0.007638  0.000384   356007.67     18        0      1     0           0 2015-11-03        -11
         -10     2015-01-01  AAK.ST  0.144321      -1.779853  0.012788  0.000419   477021.79     19        0      1     0           0 2015-11-03        -10
         -9      2015-02-01  AAK.ST  0.035877      -3.291110  0.011909  0.000223   508402.80     20        0      1     0           0 2015-11-03         -9
         -8      2015-03-01  AAK.ST  0.498285      -0.006861  0.009122  0.000464   337935.82     22        0      1     0           0 2015-11-03         -8
         -7      2015-04-01  AAK.ST  0.302305      -0.836345  0.014464  0.000605   322307.40     20        0      1     0           0 2015-11-03         -7

## DiD avec contrôles progressifs

Pour chaque événement, on estime trois spécifications cumulatives:
- M1: traitement seul
- M2: traitement + `Amihud`
- M3: traitement + `Amihud` + `Avg_Volume`

On applique la même logique aux événements ADD et DELETE, puis on répète sur les deux variables dépendantes déjà utilisées dans le pipeline, `Synchronicity` et `Idio_Vol`.

In [ ]:
did_add = build_stacked_panel(panel, matches_add, window=WINDOW_DID)
did_delete = build_stacked_panel(panel, matches_delete, window=WINDOW_DID)

print(f'ADD panel   : {len(did_add):,} rows | {did_add.index.get_level_values(0).nunique()} entities')
print(f'DELETE panel: {len(did_delete):,} rows | {did_delete.index.get_level_values(0).nunique()} entities')

for event_label, stacked_df in [('ADD', did_add), ('DELETE', did_delete)]:
    for dep_var in ['Synchronicity', 'Idio_Vol']:
        fit_panel_specs(stacked_df, dep_var, DID_CONTROL_ORDER, event_label)


## DML avec contrôles progressifs

Pour DML, on ne garde pas de modèle sans covariables: la première spécification admissible utilise le premier contrôle pré-event, puis on ajoute les autres un par un dans le même ordre.

Les datasets DML sont reconstruits à partir des prix bruts, ce qui permet de traiter ADD et DELETE de la même manière.

In [ ]:
dml_add = build_dml_dataset(matches_add, panel, 'ADD')
dml_delete = build_dml_dataset(matches_delete, panel, 'DELETE')

print(f'DML ADD    : {len(dml_add):,} rows | {int(dml_add["treated"].sum()) if len(dml_add) else 0} treated')
print(f'DML DELETE : {len(dml_delete):,} rows | {int(dml_delete["treated"].sum()) if len(dml_delete) else 0} treated')

if len(dml_add) >= 50:
    fit_dml_specs(dml_add, 'ADD')
else:
    print('Skipping DML ADD: not enough observations.')

if len(dml_delete) >= 50:
    fit_dml_specs(dml_delete, 'DELETE')
else:
    print('Skipping DML DELETE: not enough observations.')

DML ADD    : 1,268 rows | 636 treated
DML DELETE : 1,162 rows | 581 treated


/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-

DML table - ADD


/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(


,Model,Controls,Theta,SE,P_value,Sig,N,Treated
0,DML1,Log_MarketCap,0.259020,0.057929,0.000008,***,1268,636
1,DML2,"Log_MarketCap, Momentum_12m",0.129788,0.059961,0.030424,**,1268,636
2,DML3,"Log_MarketCap, Momentum_12m, Volatility_pre",0.139879,0.058341,0.016503,**,1268,636


...

/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(
/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-

DML table - DELETE


/Users/ethan.bcht/Dev/master-thesis-code/.venv/lib/python3.11/site-packages/doubleml/utils/_checks.py:194: UserWarning: Propensity predictions from learner RandomForestClassifier(n_jobs=-1, random_state=42) for ml_m are close to zero or one (eps=1e-12).
  warnings.warn(


,Model,Controls,Theta,SE,P_value,Sig,N,Treated
0,DML1,Log_MarketCap,0.160878,0.064003,0.011951,**,1162,581
1,DML2,"Log_MarketCap, Momentum_12m",0.112465,0.065741,0.087132,*,1162,581
2,DML3,"Log_MarketCap, Momentum_12m, Volatility_pre",0.121786,0.068920,0.077215,*,1162,581


Exported: /Users/ethan.bcht/Dev/master-thesis-code/data/results/regression_controls_dml_delete.csv
